In [9]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
from housepriceprediction.modeling import build_lasso_pipeline
import numpy as np
from housepriceprediction.config import DATA_RAW, RANDOM_STATE
from sklearn.model_selection import KFold, cross_val_predict
import pandas as pd
from IPython.display import display

train_df = pd.read_csv(DATA_RAW / "train.csv", keep_default_na=False, na_values=["NA"])
train_df = train_df[train_df["GrLivArea"] <= 4000].reset_index(drop=True)

y = train_df["SalePrice"].apply(np.log1p)
X = train_df.drop(columns=["Id", "SalePrice"])
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
lasso_pipeline = build_lasso_pipeline()
pred_log = cross_val_predict(lasso_pipeline, X, y, cv=cv, n_jobs=-1)
pred_price = np.expm1(pred_log)
residuals = y - pred_log
error_pct = np.expm1(residuals) * 100

errors = pd.DataFrame({
    "Id": train_df["Id"],
    "SalePrice": train_df["SalePrice"],
    "Predicted": np.round(pred_price, 2),
    "Residual": np.round(residuals, 4),
    "ErrorPct": np.round(error_pct, 2),
    "Neighborhood": train_df["Neighborhood"],
    "OverallQual": train_df["OverallQual"],
    "GrLivArea": train_df["GrLivArea"],
    "SaleCondition": train_df["SaleCondition"],
    "YearBuilt": train_df["YearBuilt"],
})

top_10_errors = errors.reindex(errors["Residual"].abs().sort_values(ascending=False).index).head(10)
print("--- Top 10 Houses with the biggest prediction errors (Out-of-Fold) ---")
display(top_10_errors)

--- Top 10 Houses with the biggest prediction errors (Out-of-Fold) ---


,Id,SalePrice,Predicted,Residual,ErrorPct,Neighborhood,OverallQual,GrLivArea,SaleCondition,YearBuilt
631,633,82500,182289.96,-0.7928,-54.74,NWAmes,7,1411,Family,1977
30,31,40000,81747.74,-0.7147,-51.07,IDOTRR,4,1317,Normal,1920
462,463,62383,119763.64,-0.6522,-47.91,Sawyer,5,864,Normal,1965
1320,1325,147000,276301.39,-0.6311,-46.80,Somerst,8,1795,Partial,2006
966,969,37900,69663.43,-0.6087,-45.59,OldTown,3,968,Abnorml,1910
88,89,85000,47992.15,0.5716,77.11,IDOTRR,3,1526,Abnorml,1915
495,496,34900,61565.19,-0.5676,-43.31,IDOTRR,4,720,Abnorml,1920
410,411,60000,100767.03,-0.5185,-40.46,Edwards,5,1276,Abnorml,1958
968,971,135000,80493.51,0.5171,67.71,NAmes,4,1192,Abnorml,1949
587,589,143000,239489.44,-0.5157,-40.29,ClearCr,5,1473,Partial,1968


In [11]:
def error_by_group(errors: pd.DataFrame, column: str, min_count: int = 1, sort_by: str = "Mean_Abs_ErrorPct", ascending: bool = False) -> pd.DataFrame:
    """Calculate error metrics grouped by a specified column.

    Allows sorting either by error metrics or linearly by group index (e.g.,
    price bands).
    """
    df = errors.copy()
    df["AbsErrorPct"] = df["ErrorPct"].abs()
    df["AbsResidual"] = df["Residual"].abs()

    grouped = (
        df.groupby(column, observed=False)
        .agg(
            Count=("Id", "count"),
            Mean_Residual=("Residual", "mean"),
            Mean_Abs_Residual=("AbsResidual", "mean"),
            Mean_Abs_ErrorPct=("AbsErrorPct", "mean"),
        )
        .reset_index()
    )
    grouped = grouped[grouped["Count"] >= min_count]
    grouped = grouped.sort_values(by=sort_by, ascending=ascending).reset_index(drop=True)
    grouped["Mean_Residual"] = grouped["Mean_Residual"].round(4)
    grouped["Mean_Abs_Residual"] = grouped["Mean_Abs_Residual"].round(4)
    grouped["Mean_Abs_ErrorPct"] = grouped["Mean_Abs_ErrorPct"].round(2)
    return grouped


errors["PriceBand"] = pd.qcut(errors["Predicted"], q=5)
errors["ActualPriceBand"] = pd.qcut(errors["SalePrice"], q=5)

predicted_band_errors = error_by_group(errors, column="PriceBand", sort_by="PriceBand", ascending=True)
actual_band_errors = error_by_group(errors, column="ActualPriceBand", sort_by="ActualPriceBand",ascending=True)

print("--- 1. CORRECT: Error Analysis by Predicted Price (PriceBand) ---")
display(predicted_band_errors)

print(
    "\n--- 2. STATISTICAL TRAP: Error Analysis by Actual Sale Price"
    " (ActualPriceBand) ---"
)

display(actual_band_errors)

--- 1. CORRECT: Error Analysis by Predicted Price (PriceBand) ---


,PriceBand,Count,Mean_Residual,Mean_Abs_Residual,Mean_Abs_ErrorPct
0,"(47992.149000000005, 123003.21]",292,-0.0055,0.1100,10.87
1,"(123003.21, 146440.77]",291,0.0100,0.0657,6.61
2,"(146440.77, 181544.2]",291,-0.0051,0.0630,6.25
3,"(181544.2, 226986.99]",291,-0.0113,0.0685,6.75
4,"(226986.99, 601009.67]",291,0.0086,0.0788,7.93



--- 2. STATISTICAL TRAP: Error Analysis by Actual Sale Price (ActualPriceBand) ---


,ActualPriceBand,Count,Mean_Residual,Mean_Abs_Residual,Mean_Abs_ErrorPct
0,"(34899.999, 124000.0]",295,-0.0469,0.1100,10.41
1,"(124000.0, 147000.0]",294,0.0039,0.0698,6.98
2,"(147000.0, 179200.0]",286,0.0080,0.0662,6.70
3,"(179200.0, 230000.0]",294,-0.0034,0.0621,6.21
4,"(230000.0, 625000.0]",287,0.0363,0.0777,8.10


In [ ]:
# Error analysis (Lasso, alpha=0.003, out-of-fold predictions)
#
# 1. No systematic bias. Mean residual is close to 0 in every group (SaleCondition,
#    Neighborhood, predicted price band): the model already uses these columns, so it has
#    learned the average price shifts between groups.
#
# 2. What differs is predictability (mean absolute error per house):
#    - SaleCondition: Normal 6.9%, Partial 9.1%, Abnorml / Family / Alloca ~13-14%.
#      Distressed or family sales depend on the seller's situation, not on the house.
#    - Neighborhood: from 4.8% (CollgCr) to 17.3% (IDOTRR). Old, mixed areas (IDOTRR,
#      OldTown, SWISU, Edwards, BrkSide) are hard; newer, homogeneous ones (CollgCr, Gilbert,
#      SawyerW) are easy.
#    - Predicted price: U-shape. Cheapest 20% -> 10.9%, middle -> ~6.5%, most expensive 20% -> 7.9%.
#
# 3. Top 10 errors: 8 of 10 overestimated, mostly Abnorml / Partial / Family sales and old,
#    cheap houses (IDOTRR, OldTown). Id 589 is "Partial" but built in 1968: possibly
#    inconsistent data.
#
# 4. Statistical trap avoided: grouping by the *actual* price shows a fake bias
#    (-4.7% for the cheapest, +3.6% for the most expensive) caused by regression to the mean.
#    Grouping by the *predicted* price (what a user sees) shows no bias.
#
# For the web app: for a normal sale, the typical error is ~7%.